## First-mode removal

### Set up and get data

In [1]:
## Import modules
import abct
import numpy as np
from scipy import io
from sklearn import cluster
import plotly.express as px
import plotly.subplots as sp

## Unpack data
data = io.loadmat("hcp_data.mat")

# Downsampled activity for fast computation
activity = data["activity"][::100]
coactivity = activity.T @ activity


### Compute variants of first-mode removal

In [2]:
## Connectivity first-mode removal
connectivity = data["connectivity"]
idx = ~np.eye(len(connectivity), dtype=bool) 

conn = dict()
conn["Degree-corrected"] = abct.moderemoval(connectivity, "degree")[idx]
conn["First-mode-removed"] = abct.moderemoval(connectivity, "rankone")[idx]

# Regress global signal and normalize to unit norm
activity_gsr = abct.moderemoval(activity.T, "global").T
activity_gsr = activity_gsr / np.linalg.norm(activity_gsr, axis=0, keepdims=True)
coactivity_gsr = activity_gsr.T @ activity_gsr

coact = dict()
coact["Degree-corrected"] = abct.moderemoval(coactivity, "degree")[idx]
coact["Global-signal-regressed"] = coactivity_gsr[idx]

### Visualize connectivity first-mode removal

In [ ]:
# Create a scatter plot of degree-corrected and rank-one corrected connectivity
r = np.corrcoef(conn["First-mode-removed"], conn["Degree-corrected"])[0, 1]
fig = px.scatter(
    x=conn["First-mode-removed"],
    y=conn["Degree-corrected"],
    labels={"x": "Link weights after first-mode removal", "y": "Link weights after degree correction"},
    title=f"Connectivity (r = {r:.3f})",
    width=600, height=600,
)
fig.update_layout(title_x=0.5).show()

### Visualize co-activity first-mode removal

In [ ]:
# Create a scatter plot of degree-corrected and global-signal regressed coactivity
r = np.corrcoef(coact["Global-signal-regressed"], coact["Degree-corrected"])[0, 1]
fig = px.scatter(
    x=coact["Global-signal-regressed"],
    y=coact["Degree-corrected"],
    labels={"x": "Link weights after global-signal regression", "y": "Link weights after degree correction"},
    title=f"Co-activity (r = {r:.3f})",
    width=600, height=600,
)
fig.update_layout(title_x=0.5).show()

## Clustering

### Set up functions

In [5]:
def kmeans_stats(X, M):
    distance = 0.0
    similarity = 0.0
    m = (X.T @ X).sum()                     # normalization factor
    for u in range(k):
        I = (M == u)                        # cluster indices
        ni = I.sum()                        # cluster size
        Xi = X[:, I]                        # cluster timeseries
        xi = Xi.mean(1, keepdims=1)         # cluster centroid
        distance += ((Xi - xi) ** 2).sum()  # distance
        similarity += (Xi.T @ Xi).sum() / ni

    return (distance / m, similarity / m)

def spectral_stats(C, M):
    similarity = 0.0
    for u in range(k):
        I = (M == u)                        # cluster indices
        similarity += C[np.ix_(I, I)].sum() / C[I].sum()

    return similarity


k = 15                  # number of clusters
repl = 100              # number of replicates

### Lloyd k-means

In [ ]:
Mk0 = cluster.KMeans(n_clusters=k, n_init=repl).fit(activity_gsr.T).labels_
print(f"Similarity (Lloyd): {kmeans_stats(activity_gsr, Mk0)[1]}")

### Loyvain k-means

In [ ]:
Mk1 = abct.loyvain(coactivity, k, "kmodularity", replicates=repl)[0]
print(f"Similarity (Loyvain): {kmeans_stats(activity_gsr, Mk1)[1]}")

### Shi-Malik spectral clustering

In [ ]:
Ms0 = cluster.SpectralClustering(n_clusters=k, affinity='precomputed', n_init=repl).fit(connectivity).labels_
print(f"Similarity (Shi-Malik): {spectral_stats(connectivity, Ms0)}")

### Loyvain spectral clustering

In [ ]:
Ms1 = abct.loyvain(connectivity, k, "spectral", replicates=repl)[0]
print(f"Similarity (Loyvain): {spectral_stats(connectivity, Ms1)}")